# 🫀 PPG Anomaly Detection using NeuroKit2 + Isolation Forest
**Dataset:** PhysioNet PPG  
**Method:** Unsupervised anomaly detection via Isolation Forest  
**Pipeline:** Load → Clean → Extract Features → Train → Visualize

## ⚙️ Step 1: Install Dependencies

In [ ]:
!pip install neurokit2 wfdb scikit-learn pandas numpy matplotlib scipy --quiet

## 📦 Step 2: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import neurokit2 as nk
import wfdb
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy.signal import welch
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

## 📥 Step 3: Load PPG Data from PhysioNet

> We use the **BIDMC PPG and Respiration Dataset** — a clean, beginner-friendly PhysioNet dataset with clearly labeled PPG signals.
> 
> 📌 Dataset: https://physionet.org/content/bidmc/1.0.0/

In [ ]:
# Download record 01 from BIDMC PPG dataset (53 subjects, 8 min each at 125 Hz)
record = wfdb.rdrecord('bidmc01', pn_dir='bidmc/1.0.0')

print('Signal names:', record.sig_name)
print('Sampling rate:', record.fs, 'Hz')
print('Signal shape:', record.p_signal.shape)

In [ ]:
# Extract the PPG channel
# BIDMC dataset: channel 0 = PLETH (PPG), channel 1 = Resp
ppg_signal = record.p_signal[:, 0]
sampling_rate = record.fs  # 125 Hz

# Quick sanity check plot
plt.figure(figsize=(14, 3))
plt.plot(ppg_signal[:sampling_rate * 10], color='steelblue', linewidth=0.8)
plt.title('Raw PPG Signal (first 10 seconds)')
plt.xlabel('Samples')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

## 🧹 Step 4: Preprocess with NeuroKit2

In [ ]:
# Clean the PPG signal (removes baseline wander + high-freq noise)
ppg_cleaned = nk.ppg_clean(ppg_signal, sampling_rate=sampling_rate)

# Plot raw vs cleaned
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(ppg_signal[:sampling_rate * 15], color='gray', linewidth=0.7)
axes[0].set_title('Raw PPG')
axes[1].plot(ppg_cleaned[:sampling_rate * 15], color='steelblue', linewidth=0.7)
axes[1].set_title('Cleaned PPG (NeuroKit2)')
plt.xlabel('Samples')
plt.tight_layout()
plt.show()

## 🔍 Step 5: Feature Extraction (Sliding Window)

We slide a **30-second window** over the signal with **50% overlap** and extract features from each window.

In [ ]:
def hrv_frequency_features(ibi):
    """Compute LF, HF power and LF/HF ratio from IBI series."""
    try:
        fs = 4.0  # resample IBI to 4 Hz
        t = np.cumsum(ibi) / 1000.0
        if t[-1] - t[0] < 10:  # need at least 10 seconds
            return {"lf_power": np.nan, "hf_power": np.nan, "lf_hf_ratio": np.nan}
        f = interp1d(t, ibi, kind='linear', fill_value='extrapolate')
        t_resampled = np.arange(t[0], t[-1], 1 / fs)
        ibi_resampled = f(t_resampled)
        freqs, psd = welch(ibi_resampled, fs=fs, nperseg=min(64, len(ibi_resampled)))
        lf = np.trapz(psd[(freqs >= 0.04) & (freqs < 0.15)])
        hf = np.trapz(psd[(freqs >= 0.15) & (freqs < 0.40)])
        return {"lf_power": lf, "hf_power": hf, "lf_hf_ratio": lf / (hf + 1e-9)}
    except Exception:
        return {"lf_power": np.nan, "hf_power": np.nan, "lf_hf_ratio": np.nan}


def extract_ppg_features(segment, sampling_rate=125):
    """Extract time-domain HRV + morphological + frequency features."""
    try:
        signals, info = nk.ppg_process(segment, sampling_rate=sampling_rate)
        peaks = info["PPG_Peaks"]

        if len(peaks) < 3:
            return None

        ibi = np.diff(peaks) / sampling_rate * 1000  # ms

        features = {
            # --- Time-domain HRV ---
            "mean_ibi":    np.mean(ibi),
            "std_ibi":     np.std(ibi),
            "rmssd":       np.sqrt(np.mean(np.diff(ibi) ** 2)),
            "pnn50":       np.sum(np.abs(np.diff(ibi)) > 50) / max(len(ibi), 1),
            "cv_ibi":      np.std(ibi) / (np.mean(ibi) + 1e-9),  # coefficient of variation

            # --- Heart Rate ---
            "mean_hr":     60000 / (np.mean(ibi) + 1e-9),
            "std_hr":      np.std(60000 / (ibi + 1e-9)),
            "min_hr":      60000 / (np.max(ibi) + 1e-9),
            "max_hr":      60000 / (np.min(ibi) + 1e-9),

            # --- Amplitude (morphology) ---
            "mean_amp":    np.mean(segment[peaks]),
            "std_amp":     np.std(segment[peaks]),

            # --- Signal statistics ---
            "skewness":    pd.Series(segment).skew(),
            "kurtosis":    pd.Series(segment).kurt(),
            "signal_range": np.ptp(segment),
        }

        # Add frequency-domain features
        freq_feats = hrv_frequency_features(ibi)
        features.update(freq_feats)

        return features

    except Exception:
        return None


print('✅ Feature extraction functions defined.')

In [ ]:
# --- Sliding window parameters ---
window_size = 30 * sampling_rate   # 30 seconds = 3750 samples
step_size   = 15 * sampling_rate   # 50% overlap

feature_list  = []
window_starts = []

total_windows = (len(ppg_cleaned) - window_size) // step_size
print(f'Total windows to process: {total_windows}')

for i, start in enumerate(range(0, len(ppg_cleaned) - window_size, step_size)):
    segment = ppg_cleaned[start : start + window_size]
    feats   = extract_ppg_features(segment, sampling_rate)
    if feats is not None:
        feature_list.append(feats)
        window_starts.append(start)

df_features = pd.DataFrame(feature_list).dropna()
print(f'\n✅ Extracted features from {len(df_features)} windows')
print(f'Feature columns: {list(df_features.columns)}')
df_features.head()

## 🌲 Step 6: Train Isolation Forest

In [ ]:
# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(df_features)

# Train Isolation Forest
# contamination = expected fraction of anomalous windows (~5%)
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    max_samples='auto'
)
iso_forest.fit(X)

# Predict: -1 = anomaly, 1 = normal
df_features['anomaly']       = iso_forest.predict(X)
df_features['anomaly_score'] = iso_forest.decision_function(X)

n_anomalies = (df_features['anomaly'] == -1).sum()
n_normal    = (df_features['anomaly'] ==  1).sum()
print(f'\n✅ Model trained!')
print(f'   Normal windows  : {n_normal}')
print(f'   Anomalous windows: {n_anomalies}')

## 📊 Step 7: Visualize Results

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# --- Plot 1: Anomaly score over time ---
axes[0].plot(df_features['anomaly_score'].values, color='steelblue', linewidth=1)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1, label='Decision boundary')
anomaly_idx = df_features['anomaly'].values == -1
axes[0].scatter(
    np.where(anomaly_idx)[0],
    df_features['anomaly_score'].values[anomaly_idx],
    color='red', s=50, zorder=5, label='Anomaly'
)
axes[0].set_title('Isolation Forest Anomaly Score per Window')
axes[0].set_xlabel('Window Index')
axes[0].set_ylabel('Score (lower = more anomalous)')
axes[0].legend()

# --- Plot 2: Mean HR over time, anomalies highlighted ---
colors = df_features['anomaly'].map({1: 'steelblue', -1: 'red'})
axes[1].scatter(
    range(len(df_features)),
    df_features['mean_hr'],
    c=colors, s=30, alpha=0.7
)
axes[1].set_title('Mean Heart Rate per Window  🔵 Normal  🔴 Anomaly')
axes[1].set_xlabel('Window Index')
axes[1].set_ylabel('Heart Rate (bpm)')

# --- Plot 3: std_ibi vs mean_hr scatter ---
axes[2].scatter(
    df_features[df_features['anomaly'] ==  1]['mean_hr'],
    df_features[df_features['anomaly'] ==  1]['std_ibi'],
    color='steelblue', alpha=0.6, label='Normal'
)
axes[2].scatter(
    df_features[df_features['anomaly'] == -1]['mean_hr'],
    df_features[df_features['anomaly'] == -1]['std_ibi'],
    color='red', alpha=0.8, label='Anomaly', s=80, marker='x'
)
axes[2].set_xlabel('Mean Heart Rate (bpm)')
axes[2].set_ylabel('IBI Std Dev (ms)')
axes[2].set_title('Feature Space: Normal vs Anomaly')
axes[2].legend()

plt.tight_layout()
plt.show()

## 🩺 Step 8: Inspect an Anomalous Window

In [ ]:
# Find the most anomalous window (lowest score)
worst_idx   = df_features['anomaly_score'].idxmin()
worst_start = window_starts[worst_idx]
worst_seg   = ppg_cleaned[worst_start : worst_start + window_size]

# Find a normal window for comparison
best_idx   = df_features['anomaly_score'].idxmax()
best_start = window_starts[best_idx]
best_seg   = ppg_cleaned[best_start : best_start + window_size]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(best_seg[:sampling_rate * 10], color='steelblue', linewidth=0.8)
axes[0].set_title('✅ Normal Window (highest score)')
axes[0].set_xlabel('Samples')

axes[1].plot(worst_seg[:sampling_rate * 10], color='red', linewidth=0.8)
axes[1].set_title('⚠️ Anomalous Window (lowest score)')
axes[1].set_xlabel('Samples')

plt.tight_layout()
plt.show()

print('\nAnomaly window features:')
print(df_features.iloc[worst_idx][['mean_ibi','std_ibi','rmssd','mean_hr','lf_hf_ratio','anomaly_score']])

## 💾 Step 9: Save Results

In [ ]:
# Save feature table with anomaly labels to CSV
df_features.to_csv('ppg_anomaly_results.csv', index=False)
print('✅ Results saved to ppg_anomaly_results.csv')
print(f'\nSummary:')
print(f'  Total windows : {len(df_features)}')
print(f'  Normal        : {(df_features["anomaly"]==1).sum()}')
print(f'  Anomalous     : {(df_features["anomaly"]==-1).sum()}')
print(f'  Anomaly rate  : {(df_features["anomaly"]==-1).mean()*100:.1f}%')

## 🔧 Tuning Guide

| Parameter | Where to change | Effect |
|---|---|---|
| `contamination` | `IsolationForest(contamination=...)` | Higher = more anomalies flagged |
| `n_estimators` | `IsolationForest(n_estimators=...)` | More trees = more stable results |
| `window_size` | `window_size = N * sampling_rate` | Larger = more stable features |
| Record number | `wfdb.rdrecord('bidmc01', ...)` | Change `01` to `02`–`53` for other subjects |